In [1]:
import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap

from scipy.interpolate import BPoly

import seaborn as sns


In [2]:
today_str = datetime.today().strftime("%Y%m%d")
today_str


'20260301'

In [3]:
start_date = '1950-01-01'
end_date = '2025-09-30'


In [4]:
custom_colors = [
    '#0000d0',    # vivid blue
    '#d00000',    # vivid red
    '#f48c06',    # bright orange
    # '#ffc300',    # strong yellow
    '#8338ec',    # vivid purple / magenta-ish
    '#50a000',    # dark green with some yellow
    #'#222222',    # charcoal black gray
    # '#3a86ff'     # strong royal blue
]

my_cmap = ListedColormap(custom_colors)


In [5]:
multpl_df = pd.read_pickle("../data/multpl_datasets_snapshot_20260216.pickle")
fred_df = pd.read_pickle("../data/fred_api_datasets_snapshot_20260216.pickle")
grok_df = pd.read_pickle("../data/grok_quarter_classifications_20260216.pickle")


In [6]:
quarterly_df = pd.concat([multpl_df, fred_df, grok_df[['market_code']] ], axis=1)
quarterly_df['market_code'] = quarterly_df['market_code'].fillna(-1).astype('int')
quarterly_df = quarterly_df.loc[start_date:end_date]
quarterly_df["div_yield2"] = quarterly_df["dividend"] / quarterly_df["sp500"]
quarterly_df["price_div"] = quarterly_df["sp500"] / quarterly_df["dividend"]
quarterly_df["price_gdp"] = quarterly_df["sp500"] / quarterly_df["gdp"]
quarterly_df["price_gdp2"] = quarterly_df["sp500"] / quarterly_df["fred_gdp"]
quarterly_df["price_gnp2"] = quarterly_df["sp500"] / quarterly_df["fred_gnp"]
quarterly_df["div_minus_baa"] = quarterly_df["div_yield"] - quarterly_df["fred_baa"] / 100.0
quarterly_df["credit_spread"] = (quarterly_df["fred_baa"] - quarterly_df["fred_aaa"]) / 100.0
quarterly_df["real_price2"] = quarterly_df["sp500"] / quarterly_df["cpi"]
quarterly_df["real_price3"] = quarterly_df["sp500"] / quarterly_df["fred_cpi"]
quarterly_df["real_price_gdp2"] = quarterly_df["sp500_adj"] / quarterly_df["gdp"]

# quarterly_df.to_csv("../data/standardized_quarterly_data_" + today_str + ".csv")
# quarterly_df.to_pickle("../data/standardized_quarterly_data_" + today_str + ".pickle")
# quarterly_df = pd.read_pickle("../data/standardized_quarterly_data_20260216.pickle")


In [7]:
vars_to_consider = [
    '10yr_ustreas',
    'credit_spread',
    'div_minus_baa',
    'fred_aaa',
    'fred_baa',
    'fred_gs10',
    'fred_tb3ms',
    'gdp_growth',
    'log_cape_shiller',
    'log_cpi',
    'log_div_yield',
    'log_div_yield2',
    'log_dividend',
    'log_earn',
    'log_earn_yld',
    'log_fed_debt',
    'log_fred_cpi',
    'log_fred_gdp',
    'log_fred_gnp',
    'log_gdp',
    'log_price_div',
    'log_price_gdp',
    'log_price_gdp2',
    'log_price_gnp2',
    'log_real_gdp',
    'log_real_price2',
    'log_real_price3',
    'log_real_price_gdp2',
    'log_sp500',
    'log_sp500_adj',
    'log_us_pop',
    'real_gdp_growth',
    'real_gdp_per_cap',
    'sp500_pe',
    'us_infl',
    'us_pop_growth'
]


In [8]:
# temp_df = pd.read_pickle("../data/standardized_quarterly_data_20260216.pickle")
# # # quarterly_df = temp_df.loc['1980-01-01'::]
# # # quarterly_df = temp_df.loc['1990-01-01'::]
# # quarterly_df = temp_df.loc['2000-01-01'::]

temp_df = quarterly_df.copy()

temp_df['log_cape_shiller']      = np.log(temp_df['cape_shiller'])
temp_df['log_cpi']               = np.log(temp_df['cpi'])
temp_df['log_div_yield']         = np.log(temp_df['div_yield'])
temp_df['log_div_yield2']        = np.log(temp_df['div_yield2'])
temp_df['log_dividend']          = np.log(temp_df['dividend'])
temp_df['log_earn']              = np.log(temp_df['earn'])
temp_df['log_earn_yld']          = np.log(temp_df['earn_yld'])
temp_df['log_fed_debt']          = np.log(temp_df['fed_debt'])
temp_df['log_fred_cpi']          = np.log(temp_df['fred_cpi'])
temp_df['log_fred_gdp']          = np.log(temp_df['fred_gdp'])
temp_df['log_fred_gnp']          = np.log(temp_df['fred_gnp'])
temp_df['log_gdp']               = np.log(temp_df['gdp'])
temp_df['log_price_div']         = np.log(temp_df['price_div'])
temp_df['log_price_gdp']         = np.log(temp_df['price_gdp'])
temp_df['log_price_gdp2']        = np.log(temp_df['price_gdp2'])
temp_df['log_price_gnp2']        = np.log(temp_df['price_gnp2'])
temp_df['log_real_gdp']          = np.log(temp_df['real_gdp'])
temp_df['log_real_price2']       = np.log(temp_df['real_price2'])
temp_df['log_real_price3']       = np.log(temp_df['real_price3'])
temp_df['log_real_price_gdp2']   = np.log(temp_df['real_price_gdp2'])
temp_df['log_sp500']             = np.log(temp_df['sp500'])
temp_df['log_sp500_adj']         = np.log(temp_df['sp500_adj'])
# temp_df['log_us_home_prices']    = np.log(temp_df['us_home_prices'])
temp_df['log_us_pop']            = np.log(temp_df['us_pop'])

quarterly_df = temp_df[vars_to_consider + ['market_code'] ].copy()


In [9]:
# original_base_columns = quarterly_df.columns.to_list()
original_base_columns = list(filter(lambda x: x != 'market_code', quarterly_df.columns.to_list()))
original_base_columns.sort()
# original_base_columns


In [10]:
for col_name in quarterly_df.columns:
    if col_name == 'market_code':
        continue

    # print(col_name)

    orig_all_dates = quarterly_df.index.values
    orig_all_date_string = [datetime.strptime(str(d), '%Y-%m-%d').date() for d in orig_all_dates]
    values = quarterly_df[col_name].values
    regimes = quarterly_df['market_code'].values
    num_nulls = int(quarterly_df[col_name].isna().sum())
    # print("min date = " + str(orig_all_date_string[0]))
    # print("max date = " + str(orig_all_date_string[-1]))
    # print("num values = " + str(int(quarterly_df[col_name].notna().sum())) )
    # print("num nulls " + str(num_nulls) )

    if num_nulls > 0:

        orig_values = quarterly_df[[col_name, 'market_code']]
        temp_df = orig_values.dropna()
        date_strings = temp_df.index.values
        dates = [datetime.strptime(str(d), '%Y-%m-%d').date() for d in date_strings]

        values = temp_df[col_name].values
        regimes = temp_df['market_code'].values
        smoothed_centered = temp_df[col_name].rolling(window=5, min_periods=1, center=True).mean()
        smoothed_dates_numeric = mdates.date2num(dates) - mdates.date2num(dates).min()

        dy_dx = np.gradient(smoothed_centered, smoothed_dates_numeric)
        derivative = pd.Series(dy_dx, index=dates)
        derivative_smoothed = derivative.rolling(window=5, min_periods=1, center=True).mean()

        second_der_smoothed = pd.Series(np.gradient(derivative_smoothed, smoothed_dates_numeric), index=dates).rolling(
            window=5, min_periods=1, center=True).mean()

        third_der_smoothed = pd.Series(np.gradient(second_der_smoothed, smoothed_dates_numeric), index=dates).rolling(
            window=5, min_periods=1, center=True).mean()

        t = (mdates.date2num(orig_all_dates) - mdates.date2num(orig_all_dates).min()).astype("int64")
        y = orig_values[col_name].reindex(orig_all_dates).copy()
        d1 = derivative_smoothed.reindex(orig_all_date_string).copy()
        d2 = second_der_smoothed.reindex(orig_all_date_string).copy()
        d3 = third_der_smoothed.reindex(orig_all_date_string).copy()

        m = y.isna().values
        edges = np.flatnonzero(np.diff(np.r_[0, m, 0]))

        for s, e in edges.reshape(-1,2):
            L, R = s-1, e
            if L>=0 and R<len(y):
                p = BPoly.from_derivatives([t[L],t[R]],[[y.iloc[L],d1.iloc[L],d2.iloc[L],d3.iloc[L]],
                                                       [y.iloc[R],d1.iloc[R],d2.iloc[R],d3.iloc[R]]])
                y.iloc[s:R] = p(t[s:R])
            elif R<len(y):
                dt = t[s:R]-t[R]
                y.iloc[s:R] = y.iloc[R]+d1.iloc[R]*dt+d2.iloc[R]*dt**2/2+d3.iloc[R]*dt**3/6
            else:
                dt = t[L+1:]-t[L]
                y.iloc[L+1:] = y.iloc[L]+d1.iloc[L]*dt+d2.iloc[L]*dt**2/2+d3.iloc[L]*dt**3/6

        # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(5, 12), sharex=True)

        # fig.suptitle(col_name)

        # ax1.scatter(
        #     x=dates, y=values,
        #     c=regimes, cmap=my_cmap, # good maps include 'viridis' 'Set1' 'tab10' 'Accent' 'rainbow' 'turbo'
        #     s=10, # use 20 for a 8x5 plot, but 10 is better for smaller plots
        #     alpha=0.7,
        #     linestyle='solid', marker='o')
        # # plt.yscale('log')
        # formatter = mdates.DateFormatter('%Y-%m-%d')
        # ax1.xaxis.set_major_formatter(formatter)
        # fig.autofmt_xdate()

        # ax2.scatter(
        #     x=dates, y=derivative_smoothed,
        #     c=regimes, cmap=my_cmap,
        #     s=10,
        #     alpha=0.7,
        #     linestyle='solid', marker='o')

        # ax3.scatter(
        #     x=dates, y=second_der_smoothed,
        #     c=regimes, cmap=my_cmap,
        #     s=10,
        #     alpha=0.7,
        #     linestyle='solid', marker='o')

        # ax4.scatter(
        #     x=dates, y=third_der_smoothed,
        #     c=regimes, cmap=my_cmap,
        #     s=10,
        #     alpha=0.7,
        #     linestyle='solid', marker='o')

        # ax5.scatter(
        #     x=orig_all_date_string, y=y,
        #     c=quarterly_df['market_code'].values, cmap=my_cmap,
        #     s=10,
        #     alpha=0.7,
        #     linestyle='solid', marker='o')

        # plt.tight_layout()
        # plt.show()

        # overwrite the original column NaNs with the inferred values
        quarterly_df[col_name] = y

    # print()
    # print()


In [11]:
for col_name in quarterly_df.columns:
    if col_name == 'market_code':
        continue

    # print(col_name)

    orig_all_dates = quarterly_df.index.values
    orig_all_date_string = [datetime.strptime(str(d), '%Y-%m-%d').date() for d in orig_all_dates]
    values = quarterly_df[col_name].values
    regimes = quarterly_df['market_code'].values
    num_nulls = int(quarterly_df[col_name].isna().sum())
    
    # print("min date = " + str(orig_all_date_string[0]))
    # print("max date = " + str(orig_all_date_string[-1]))
    # print("num values = " + str(int(quarterly_df[col_name].notna().sum())) )
    # print("num nulls " + str(num_nulls) )

    orig_values = quarterly_df[[col_name, 'market_code']]
    temp_df = orig_values.dropna()
    date_strings = temp_df.index.values
    dates = [datetime.strptime(str(d), '%Y-%m-%d').date() for d in date_strings]

    values = temp_df[col_name].values
    regimes = temp_df['market_code'].values
    smoothed_centered = temp_df[col_name].rolling(window=5, min_periods=1, center=True).mean()
    smoothed_dates_numeric = mdates.date2num(dates) - mdates.date2num(dates).min()

    dy_dx = np.gradient(smoothed_centered, smoothed_dates_numeric)
    derivative = pd.Series(dy_dx, index=dates)
    derivative_smoothed = derivative.rolling(window=5, min_periods=1, center=True).mean()

    second_der_smoothed = pd.Series(np.gradient(derivative_smoothed, smoothed_dates_numeric), index=dates).rolling(
        window=5, min_periods=1, center=True).mean()

    third_der_smoothed = pd.Series(np.gradient(second_der_smoothed, smoothed_dates_numeric), index=dates).rolling(
        window=5, min_periods=1, center=True).mean()


    # fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(5, 12), sharex=True)

    # fig.suptitle(col_name)

    # ax1.scatter(
    #     x=dates, y=values,
    #     c=regimes, cmap=my_cmap, # good maps include 'viridis' 'Set1' 'tab10' 'Accent' 'rainbow' 'turbo'
    #     s=10, # use 20 for a 8x5 plot, but 10 is better for smaller plots
    #     alpha=0.7,
    #     linestyle='solid', marker='o')
    # # plt.yscale('log')
    # formatter = mdates.DateFormatter('%Y-%m-%d')
    # ax1.xaxis.set_major_formatter(formatter)
    # fig.autofmt_xdate()

    # ax2.scatter(
    #     x=dates, y=derivative_smoothed,
    #     c=regimes, cmap=my_cmap,
    #     s=10,
    #     alpha=0.7,
    #     linestyle='solid', marker='o')

    # ax3.scatter(
    #     x=dates, y=second_der_smoothed,
    #     c=regimes, cmap=my_cmap,
    #     s=10,
    #     alpha=0.7,
    #     linestyle='solid', marker='o')

    # ax4.scatter(
    #     x=dates, y=third_der_smoothed,
    #     c=regimes, cmap=my_cmap,
    #     s=10,
    #     alpha=0.7,
    #     linestyle='solid', marker='o')

    # plt.tight_layout()
    # plt.show()


    quarterly_df[col_name+'_d1'] = derivative_smoothed.values
    quarterly_df[col_name+'_d2'] = second_der_smoothed.values
    quarterly_df[col_name+'_d3'] = third_der_smoothed.values
    
    # if col_name in ('10yr_ustreas', '):
    #     quarterly_df[colname+'_d2'] = second_der_smoothed

    # get rid of the "DataFrame is highly fragmented" warnings
    # from multiple frame.insert operations
    quarterly_df = quarterly_df.copy()
    
    # print()
    # print()


In [12]:
new_base_columns = list(filter(lambda x: x != 'market_code', quarterly_df.columns.to_list()))
new_base_columns.sort()
# new_base_columns


In [13]:
prepared_quarterly_df = quarterly_df[new_base_columns + ['market_code']].copy()

In [14]:
prepared_quarterly_df.to_csv("../data/prepared_quarterly_data_smoothed_" + today_str + ".csv")
prepared_quarterly_df.to_pickle("../data/prepared_quarterly_data_smoothed_" + today_str + ".pickle")
